In [ ]:
import time
import warnings

import fastf1 as ff1
import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# --- Configuration ---
INPUT_FILE = "/workspaces/drs_data/data/master_qualifying_data_2023-2026.csv"
OUTPUT_FILE = "/workspaces/drs_data/data/master_qualifying_data_enriched.csv"

print(f"Loading input data: {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"✅ Data loaded: {len(df)} rows.")
except FileNotFoundError:
    print("🚨 Input file not found.")
    raise

In [ ]:
# helper 

def get_standings_before_round(year, round_num):
    """
    Fetches driver/constructor standings AFTER round (n-1).
    If round_num is 1, returns empty/zero stats.
    """
    if round_num <= 1:
        return {}, {}  # Start of season, everyone is 0

    # We want standings BEFORE this race, so we ask for round - 1
    prev_round = int(round_num) - 1
    url = f"http://api.jolpi.ca/ergast/f1/{year}/{prev_round}/driverStandings.json"

    try:
        # Polite request with timeout
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()

        driver_stats = {}

        # Parse Driver Standings
        try:
            standings_list = data["MRData"]["StandingsTable"]["StandingsLists"][0][
                "DriverStandings"
            ]
            for item in standings_list:
                code = item["Driver"]["code"]  # e.g., "VER"
                points = float(item["points"])
                wins = int(item["wins"])
                position = int(item["position"])

                driver_stats[code] = {
                    "Points": points,
                    "Wins": wins,
                    "Champ_Pos": position,
                }
        except (KeyError, IndexError):
            print(f"  ⚠️ No standings data found for {year} Round {prev_round}")

        # (We could also fetch Constructor standings here similarly)

        return driver_stats

    except Exception as e:
        print(f"  🚨 API Error ({year} R{prev_round}): {e}")
        return {}

In [ ]:
# map round numbers
print("Mapping events to Round Numbers using FastF1...")
# We need to know that "Bahrain Grand Prix" is Round 1
ff1.Cache.enable_cache("/workspaces/f1-project/cache")

# Create a mapping dictionary: (Year, EventName) -> RoundNumber
event_to_round = {}

unique_years = df["Year"].unique()
for year in unique_years:
    schedule = ff1.get_event_schedule(year, include_testing=False)
    for _, row in schedule.iterrows():
        event_to_round[(year, row["EventName"])] = row["RoundNumber"]

# Apply mapping to dataframe
df["RoundNumber"] = df.apply(
    lambda x: event_to_round.get((x["Year"], x["Event"]), 0), axis=1
)
print("✅ Round numbers mapped.")

In [ ]:
# enrichment loop 

print("Fetching Championship Standings from Jolpica...")

# Create new columns
df["Driver_Points"] = 0.0
df["Driver_Wins"] = 0
df["Championship_Pos"] = 0

# Get unique Race instances (Year + Round) to minimize API calls
unique_races = (
    df[["Year", "RoundNumber"]].drop_duplicates().sort_values(["Year", "RoundNumber"])
)

for _, race in tqdm(
    unique_races.iterrows(), total=len(unique_races), desc="Fetching Seasons"
):
    year = race["Year"]
    round_num = race["RoundNumber"]

    if round_num == 0:
        continue  # Skip unrecognized races

    # Fetch API Data
    stats = get_standings_before_round(year, round_num)

    # Update DataFrame for this specific race
    # We filter rows matching this Year and Round
    mask = (df["Year"] == year) & (df["RoundNumber"] == round_num)

    # Map the API data to the rows
    # We iterate through the drivers in the current dataframe slice
    for idx in df[mask].index:
        driver_code = df.loc[idx, "Driver"]  # e.g., "VER"

        # Some drivers might have different codes or be rookies (not in standings yet)
        if driver_code in stats:
            df.loc[idx, "Driver_Points"] = stats[driver_code]["Points"]
            df.loc[idx, "Driver_Wins"] = stats[driver_code]["Wins"]
            df.loc[idx, "Championship_Pos"] = stats[driver_code]["Champ_Pos"]
        else:
            # Rookies or first race of season -> 0 points, position 20 (default)
            df.loc[idx, "Driver_Points"] = 0
            df.loc[idx, "Driver_Wins"] = 0
            df.loc[idx, "Championship_Pos"] = 20

    # Be polite to the API (small delay)
    time.sleep(0.2)

In [ ]:
# save dataset

df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Enriched dataset saved to: {OUTPUT_FILE}")
print("Sample Data (with Points):")
print(df[["Year", "Event", "Driver", "Driver_Points", "Championship_Pos"]].sample(5))